# Laboratorio 2 — Limpieza de datos
### Dataset: `retail_ferreteria.csv` — Tópicos Especiales II, UTP

Una respuesta por cada punto de la actividad.

In [ ]:
import pandas as pd, numpy as np, math
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')

raw = pd.read_csv('retail_ferreteria.csv', encoding='utf-8')
df = raw.copy()          # df = copia de trabajo que se irá limpiando
print('Cargado:', raw.shape)

---
## Punto 2 — Análisis exploratorio

In [ ]:
print('--- Numéricas ---')
print(raw[['cantidad','precio_unitario','total_venta']].describe().to_string())

print('\n--- Categóricas ---')
for c in ['sucursal','categoria','medio_pago','promo_producto','promo_pago']:
    print(f'{c} ({raw[c].nunique()} valores):',
          dict(raw[c].value_counts(dropna=False).head(8)))

print('\n--- Coherencia total_venta = cantidad * precio_unitario ---')
comp = raw.dropna(subset=['cantidad','precio_unitario'])
print('filas incoherentes:', int(((comp.cantidad*comp.precio_unitario - comp.total_venta).abs() > 0.01).sum()))
print('totales negativos :', int((raw.total_venta < 0).sum()))

### Respuesta

El dataset tiene **3 150 ventas y 15 columnas** de una cadena de 4 ferreterías (una por
provincia), con 10 productos en 7 categorías, entre sep-2024 y sep-2025.

La exploración detectó **4 problemas**: `total_venta` tiene un mínimo de **−220 383** (imposible
en una venta), `promo_producto` contiene el valor **`???`**, hay **nulos** en cantidad, precio y
medio de pago, y hay **duplicados**. Las 19 filas incoherentes son exactamente las de total
negativo: su valor absoluto sí cuadra con `cantidad × precio`, o sea que es un **error de signo**.

---
## Punto 3 — Análisis exploratorio inicial (`shape`, `info`, `isna`, `duplicated`)

In [ ]:
print('=== SHAPE ===');      print(raw.shape)
print('\n=== INFO ===');     raw.info()
print('\n=== ISNA ===');     print(raw.isna().sum().to_string())
print('\n=== DUPLICATED ===')
print('filas duplicadas exactas:', int(raw.duplicated().sum()))
print('id_venta repetidos      :', int(raw.duplicated(subset='id_venta').sum()))

### Respuesta

| Comprobación | Resultado |
|---|---|
| `shape` | 3 150 filas × 15 columnas |
| `info()` | `fecha_hora` es texto (debería ser fecha) y `cantidad` es `float` (debería ser entero) |
| `isna().sum()` | 5 460 celdas nulas en 6 columnas |
| `duplicated()` | 135 filas idénticas y 150 `id_venta` repetidos |

Dos detalles que marcan el resto del trabajo: los **150 duplicados están todos al final** del
archivo (índices 3000-3149), o sea que son una reinyección de filas anteriores; y como 150 ≠ 135,
hay **15 filas que comparten `id_venta` pero difieren en algún valor**.

---
## Punto 4 — Manejo de valores nulos

In [ ]:
# Los nulos de cantidad/precio se DESPEJAN de total = cantidad * precio (no se estiman).
# Se usa abs() porque el error de signo todavía no se corrige (eso va en el punto 6).
m = df.precio_unitario.isna()
df.loc[m,'precio_unitario'] = (df.loc[m,'total_venta'].abs() / df.loc[m,'cantidad']).round(2)

m = df.cantidad.isna()
derivada = df.loc[m,'total_venta'].abs() / df.loc[m,'precio_unitario']
print('¿Las cantidades derivadas dan enteros?  desvío máx =',
      float((derivada.round()-derivada).abs().max()), ' rango =',
      float(derivada.min()), '-', float(derivada.max()))
df.loc[m,'cantidad'] = derivada.round()
df['cantidad'] = df.cantidad.astype('int64')

# Nulos que NO se pueden derivar
df['medio_pago']     = df.medio_pago.fillna('no_informado')
# Nulos estructurales: "no hubo promoción" / "no hubo productos adjuntos"
df['promo_producto'] = df.promo_producto.fillna('sin_promo')
df['promo_pago']     = df.promo_pago.fillna('sin_promo')
df['otros_productos']= df.otros_productos.fillna('')

print('\nNulos restantes:', int(df.isna().sum().sum()))

### Respuesta

**Cómo se limpió:** los nulos se separaron en tres tipos y cada uno recibió un tratamiento distinto.

1. **`cantidad` (59) y `precio_unitario` (60) → derivación exacta.** `total_venta` está completa y
   se cumple `total = cantidad × precio`, así que el valor faltante se despeja dividiendo. **No se
   imputó por media ni mediana**: la prueba de que la derivación es correcta es que las 59
   cantidades derivadas dan **enteros exactos entre 1 y 5** (desvío 8,9e-16), el mismo rango del
   resto de la columna.
2. **`medio_pago` (59) → categoría `no_informado`.** No se puede deducir de ninguna otra columna.
   No se borraron las filas (son ventas válidas) ni se imputó la moda, que habría inflado un medio
   de pago y arruinado el reporte 4.
3. **`promo_producto` (1 533), `promo_pago` (1 544) y `otros_productos` (2 205) → no son errores.**
   Significan "esta venta no tuvo promoción / no llevó productos adjuntos". Se rellenaron con
   `sin_promo` y cadena vacía. Borrar esas filas habría eliminado el 70 % del dataset.

Resultado: **0 nulos**, sin perder ninguna fila.

---
## Punto 5 — Detección y tratamiento de duplicados

In [ ]:
print('Antes:', len(df), 'filas')

# 1) Duplicados exactos (fila idéntica en las 15 columnas)
df = df.drop_duplicates()
print('Tras eliminar duplicados exactos:', len(df))

# 2) id_venta repetidos con contenido distinto: se conserva la copia más informativa
df['_incompleto'] = (df.medio_pago == 'no_informado').astype(int)
df = (df.sort_values(['id_venta','_incompleto'])
        .drop_duplicates(subset='id_venta', keep='first')
        .drop(columns='_incompleto')
        .reset_index(drop=True))

print('Tras consolidar id_venta   :', len(df))
print('\nVERIFICACIÓN')
print('  duplicados exactos :', int(df.duplicated().sum()))
print('  id_venta repetidos :', int(df.duplicated(subset="id_venta").sum()))
print('  id_venta únicos    :', df.id_venta.nunique(), '(los 3000 originales:',
      set(df.id_venta) == set(raw.id_venta), ')')

### Respuesta

**Cómo se limpió:** en dos pasos, porque había dos tipos de duplicado.

1. **135 filas exactamente idénticas** → se eliminaron con `drop_duplicates()`, conservando la
   primera. Comparten `id_venta`, cliente, fecha al segundo y monto: es el mismo registro cargado
   dos veces, no dos ventas.
2. **15 `id_venta` repetidos con contenido distinto** → aquí no se podía borrar a ciegas. Al
   revisarlos, las dos versiones eran **complementarias**: donde una copia tenía un nulo, la otra
   tenía el dato bueno. Como el punto 4 ya rellenó esos nulos, las copias quedaron casi iguales;
   para las que aún diferían se conservó **la que tiene el medio de pago informado**, no una al azar.

Resultado: **3 150 → 3 000 filas**. Se eliminaron 150 duplicados y **no se perdió ninguna venta**:
los 3 000 `id_venta` originales siguen presentes.

---
## Punto 6 — Formatos inconsistentes, valores atípicos o inválidos

In [ ]:
# a) Tipos de dato
df['fecha_hora'] = pd.to_datetime(df.fecha_hora)

# b) Texto: espacios sobrantes y tratamientos honoríficos (Sr(a). / Dr(a).)
COLS_TXT = ['nombre_cliente','sucursal','provincia','producto','categoria',
            'medio_pago','promo_producto','promo_pago','otros_productos']
for c in COLS_TXT:
    df[c] = df[c].astype('string').str.strip().str.replace(r'\s+',' ', regex=True)
df['nombre_cliente'] = df.nombre_cliente.str.replace(
    r'^(?:Sr|Sra|Dr|Dra|Ing|Lic)(?:\(a\))?\.\s*', '', regex=True).str.strip()

# c) Valores inválidos
n_neg  = int((df.total_venta < 0).sum())
n_int  = int((df.promo_producto == '???').sum())
df['total_venta']    = df.total_venta.abs()                      # error de signo
df['promo_producto'] = df.promo_producto.replace('???','sin_promo')  # marcador, no categoría

print(f'En el dataset ORIGINAL: {int((raw.total_venta<0).sum())} totales negativos, '
      f'{int((raw.promo_producto=="???").sum())} valores "???"')
print(f'Corregidos aquí (tras quitar duplicados): {n_neg} negativos, {n_int} valores "???"')
print('\nVERIFICACIÓN')
print('  negativos            :', int((df.total_venta < 0).sum()))
print('  valores "???"        :', int((df.promo_producto == '???').sum()))
print('  honoríficos          :', int(df.nombre_cliente.str.contains(r'\(a\)\.').sum()))
print('  espacios dobles      :', int(df.nombre_cliente.str.contains('  ').sum()))
print('  promociones válidas  :', sorted(df.promo_producto.unique()))

# d) Valores atípicos (regla de Tukey)
q1, q3 = df.total_venta.quantile([.25,.75]); iqr = q3-q1
at = (df.total_venta < q1-1.5*iqr) | (df.total_venta > q3+1.5*iqr)
print(f'\nAtípicos por IQR en total_venta: {int(at.sum())} '
      f'| todos con cantidad = {sorted(df[at].cantidad.unique())} '
      f'| ¿coherentes?: {bool(((df[at].cantidad*df[at].precio_unitario - df[at].total_venta).abs()<0.51).all())}')

### Respuesta

**Cómo se limpió:**

| Problema | Cantidad | Decisión |
|---|---|---|
| `fecha_hora` como texto, `cantidad` como decimal | — | Convertidas a `datetime` y entero |
| Nombres con `Sr(a).` / `Dr(a).` | 618 | Se quitó el tratamiento: es formato, no identidad |
| Nombres con espacios dobles | 21 | Colapsados a un espacio |
| `total_venta` negativo | 20 | **Error de signo** → valor absoluto |
| `promo_producto` = `???` | 15 | Marcador de dato desconocido → `sin_promo` |

Las cantidades son las del dataset original; la celda corrige 19 y 13 porque el resto ya
desapareció al eliminar los duplicados en el punto 5.

Los negativos **no** se borraron como devoluciones: se comprobó que su valor absoluto coincide
exactamente con `cantidad × precio`, y el dataset no tiene ninguna columna de devolución.

**Atípicos:** la regla de Tukey marca 47 ventas, pero **se conservaron todas**. No son errores:
las 47 tienen `cantidad = 5` y un precio alto, o sea que son el techo aritmético normal del
dataset (5 × ~50 000 ≈ 250 000). Además `precio_unitario` no tiene ningún atípico. Borrarlas
habría sesgado los ingresos a la baja eliminando justo las ventas más grandes.

Las categóricas cerradas (sucursal, provincia, categoría, producto, medio de pago) ya estaban
bien: la cantidad de valores distintos no cambia al normalizar mayúsculas y espacios.

---
## Punto 7 — Hallazgos: qué encontramos, cómo lo detectamos, qué decidimos

| # | Qué encontramos | Cómo lo detectamos | Qué decidimos |
|---|---|---|---|
| 1 | 135 filas idénticas, todas al final del archivo | `duplicated().sum()` + índice | Eliminarlas |
| 2 | 15 `id_venta` repetidos con versiones complementarias | 150 repetidos ≠ 135 exactos | Conservar la copia con el dato informado |
| 3 | 20 `total_venta` negativos | Mínimo de `describe()` = −220 383 | Error de signo → valor absoluto |
| 4 | 59 cantidades y 60 precios nulos | `isna().sum()` | Despejar de `total = cantidad × precio` |
| 5 | 59 `medio_pago` nulos, no derivables | `isna().sum()` | Categoría `no_informado` |
| 6 | `???` en `promo_producto` | `value_counts(dropna=False)` | Marcador de dato faltante → `sin_promo` |
| 7 | 618 nombres con `Sr(a).`/`Dr(a).` y 21 con espacios dobles | `str.contains()` | Normalizar el texto |
| 8 | Hasta 70 % de nulos en promociones y adjuntos | `isna()` + significado de negocio | **No son errores**: rellenar, no borrar |
| 9 | 47 atípicos por IQR en `total_venta` | Regla de Tukey | Conservar: son coherentes |
| 10 | `id_cliente` no identifica personas | Cruce id ↔ nombre (ver reporte 2) | No es corregible: se documenta |

**Por qué este orden.** Los nulos se resolvieron **antes** que los duplicados porque las copias
en conflicto se completan entre sí, y la corrección de valores inválidos fue **al final** porque
así el signo y los `???` quedan arreglados en la única copia que sobrevive. Los atípicos se
evaluaron de últimos: hacerlo antes de corregir los negativos habría marcado como atípicos los
20 errores de signo.

---
## Punto 8 — Exportación del dataset limpio con verificación

In [ ]:
df['n_otros'] = df.otros_productos.apply(lambda s: 0 if not s else len(s.split(',')))

pruebas = {
 'Sin duplicados exactos'          : df.duplicated().sum() == 0,
 'id_venta único'                  : df.id_venta.is_unique,
 'No se perdió ninguna venta'      : set(df.id_venta) == set(raw.id_venta),
 'Sin nulos'                       : df.isna().sum().sum() == 0,
 'Sin totales negativos'           : (df.total_venta > 0).all(),
 'total = cantidad * precio'       : ((df.cantidad*df.precio_unitario - df.total_venta).abs() < 0.51).all(),
 'cantidad entera de 1 a 5'        : df.cantidad.between(1,5).all() and df.cantidad.dtype.kind=='i',
 'Sin marcadores "???"'            : not (df[COLS_TXT]=='???').any().any(),
 'Sin honoríficos'                 : not df.nombre_cliente.str.contains(r'\(a\)\.').any(),
 'fecha_hora es datetime'          : pd.api.types.is_datetime64_any_dtype(df.fecha_hora),
 'Ingresos coherentes con origen'  : abs(df.total_venta.sum() -
                                     raw.drop_duplicates('id_venta').total_venta.abs().sum()) < 1,
}
for k,v in pruebas.items():
    print(f'  [{"OK" if v else "FALLA"}] {k}')
assert all(pruebas.values()), 'La limpieza no tuvo el efecto esperado'

df.to_csv('retail_ferreteria_limpio.csv', index=False, encoding='utf-8-sig')

# Se relee el archivo para confirmar que sobrevivió el guardado
chk = pd.read_csv('retail_ferreteria_limpio.csv', encoding='utf-8-sig', parse_dates=['fecha_hora'])
print(f'\nExportado retail_ferreteria_limpio.csv -> {chk.shape[0]:,} filas x {chk.shape[1]} columnas')
print('  nulos al releer      :', int(chk.isna().sum().sum() - chk.otros_productos.isna().sum()))
print('  ingresos coinciden   :', abs(chk.total_venta.sum() - df.total_venta.sum()) < 0.01)
print('  acentos preservados  :', [c for c in sorted(chk.categoria.unique()) if 'ó' in c or 'é' in c])

### Respuesta

El dataset limpio se exportó a **`retail_ferreteria_limpio.csv`** (3 000 filas × 16 columnas, en
UTF-8 con BOM para que Excel muestre bien los acentos).

**Verificación:** se corrieron **11 pruebas automáticas** con `assert`, de modo que el notebook
falla antes de exportar si algún paso no tuvo el efecto esperado. Además el archivo **se volvió a
leer desde el disco** para confirmar que los montos, los tipos y los acentos sobrevivieron al
guardado.

| | Original | Limpio |
|---|---|---|
| Filas | 3 150 | 3 000 |
| Duplicados | 135 exactos / 150 por `id_venta` | 0 |
| Celdas nulas | 5 460 | 0 |
| Totales negativos | 20 | 0 |
| Valores `???` | 15 | 0 |
| Nombres con honorífico | 618 | 0 |

Los ingresos totales coinciden con los del dataset original una vez corregido el signo, lo que
confirma que **no se perdió ninguna venta** en la limpieza.

---
## Punto 9 — Cuatro reportes sobre el dataset limpio

In [ ]:
def welch(a, b):
    # Prueba t de Welch. El entorno no tiene scipy; con n>500 la t se aproxima con la normal.
    a, b = np.asarray(a,float), np.asarray(b,float)
    se = math.sqrt(a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b))
    t = (a.mean()-b.mean())/se
    return t, 2*(1 - 0.5*(1+math.erf(abs(t)/math.sqrt(2))))

def comparar(titulo, a, b, na, nb):
    t, p = welch(a, b)
    fmt = ',.0f' if np.mean(b) > 100 else ',.2f'
    print(f'{titulo}\n  {na}: {np.mean(a):{fmt}}   {nb}: {np.mean(b):{fmt}}   '
          f'dif: {(np.mean(a)/np.mean(b)-1)*100:+.1f}%   p={p:.4f}  -> '
          f'{"SIGNIFICATIVO" if p<0.05 else "no significativo"}')
    return p
print('Listo.')

### Reporte 1 — La diferencia entre sucursales es de tráfico, no de ticket

In [ ]:
r1 = (df.groupby('sucursal')
        .agg(transacciones=('id_venta','count'), ingresos=('total_venta','sum'),
             ticket=('total_venta','mean'))
        .sort_values('ingresos', ascending=False))
print(r1.to_string())

mejor, peor = r1.index[0], r1.index[-1]
print(f'\nBrecha de ingresos {mejor} vs {peor}: '
      f'{(r1.loc[mejor,"ingresos"]/r1.loc[peor,"ingresos"]-1)*100:+.1f}%')
print(f'  por transacciones: {(r1.loc[mejor,"transacciones"]/r1.loc[peor,"transacciones"]-1)*100:+.1f}%')
print(f'  por ticket       : {(r1.loc[mejor,"ticket"]/r1.loc[peor,"ticket"]-1)*100:+.1f}%')
print(f'\nDispersión del ticket entre las 4 sucursales: {(r1.ticket.max()/r1.ticket.min()-1)*100:.1f}%\n')
comparar('Ticket Córdoba vs Rosario',
         df.loc[df.sucursal==mejor,'total_venta'], df.loc[df.sucursal==peor,'total_venta'], mejor, peor)

**Qué se encontró:** Córdoba factura **21,9 % más** que Rosario, pero el ticket promedio de las
4 sucursales es prácticamente igual (dispersión de 3,6 %). Toda la brecha viene del **número de
transacciones**: 794 contra 670.

**Cómo se comprobó:** descomponiendo `ingresos = transacciones × ticket` (el tráfico explica el
87 % de la brecha) y con una prueba t de Welch sobre el ticket, que da **p = 0,50**: la diferencia
de ticket es ruido.

**Recomendación:** la palanca en Rosario es **captación, no upselling** — capacitar en venta
adicional tiene techo bajo porque el ticket ya está al nivel de la red. Igualar el volumen de
Córdoba vale **≈ 9,2 M** de ingresos. El KPI a seguir debe ser *transacciones por día*, no ticket
promedio, que al ser homogéneo no sirve para gestionar.

### Reporte 2 — El `id_cliente` no identifica clientes

In [ ]:
print('Ventas                                   :', len(df))
print('Pares (id_cliente, nombre_cliente) únicos:', df.drop_duplicates(['id_cliente','nombre_cliente']).shape[0])
print('Ventas que repiten el MISMO par          :', len(df) - df.drop_duplicates(['id_cliente','nombre_cliente']).shape[0])

n_compras = df.groupby('id_cliente').size()
n_nombres = df.groupby('id_cliente').nombre_cliente.nunique()
print(f'\nid_cliente con más de una compra: {int((n_compras>1).sum())}')
print(f'id_cliente con más de un nombre : {int((n_nombres>1).sum())}')
print(f'¿Son el mismo conjunto?: {set(n_compras[n_compras>1].index)==set(n_nombres[n_nombres>1].index)}')

# Si el id se sorteara al azar, ¿cuántas repeticiones habría? (paradoja del cumpleaños)
rango = int(df.id_cliente.max()-df.id_cliente.min()+1)
esperadas = len(df) - rango*(1-(1-1/rango)**len(df))
print(f'\nColisiones esperadas por azar puro: {esperadas:.1f}')
print(f'Colisiones observadas             : {len(df)-df.id_cliente.nunique()}')

print('\nEjemplo de un id compartido por personas distintas:')
ej = n_nombres[n_nombres>1].index[0]
print(df[df.id_cliente==ej][['id_cliente','nombre_cliente','fecha_hora']].to_string(index=False))

**Qué se encontró:** a primera vista hay 2 552 clientes con 1,18 compras cada uno y 399 clientes
"recurrentes". **Ninguna de esas cifras es real.** En las 3 000 ventas hay **3 000 pares
(`id_cliente`, `nombre_cliente`) distintos**: ni una sola vez se repite el mismo id con el mismo
nombre. Los 399 "recurrentes" son ids que aparecen con **un nombre diferente en cada compra**.

**Cómo se comprobó:** contando pares únicos, y contra el azar: si el id se sorteara al azar en su
rango (1001–9991), la paradoja del cumpleaños predice **449,1 repeticiones**; se observan **448**.
Es indistinguible del azar.

**Recomendación:** **suspender todo indicador basado en `id_cliente`** — no se puede calcular
recompra, valor de vida ni concentración de cartera, y por lo tanto no se puede lanzar un programa
de fidelización, porque no habría forma de medir si funciona. Hay que **implementar una identidad
real en el punto de venta** (documento, teléfono o correo). Mientras tanto, el negocio se gestiona
por transacciones, no por clientes.

### Reporte 3 — Las promociones no mueven el ticket

In [ ]:
print(df.groupby('promo_producto')
        .agg(ventas=('id_venta','count'), ticket=('total_venta','mean'),
             unidades=('cantidad','mean')).to_string())

ctrl = df.loc[df.promo_producto=='sin_promo','total_venta']
print(f'\nVentas con promoción: {int((df.promo_producto!="sin_promo").sum()):,} '
      f'({(df.promo_producto!="sin_promo").mean()*100:.1f}%)\n')

comparar('Ticket: con promo vs sin promo',
         df.loc[df.promo_producto!='sin_promo','total_venta'], ctrl, 'con promo', 'sin promo')
for p in ['2x1','10% off','sin interés']:
    comparar(f'Ticket: {p} vs control', df.loc[df.promo_producto==p,'total_venta'], ctrl, p, 'control')
comparar('Unidades: 2x1 vs control (un 2x1 DEBERÍA moverlas)',
         df.loc[df.promo_producto=='2x1','cantidad'], df.loc[df.promo_producto=='sin_promo','cantidad'],
         '2x1', 'control')

**Qué se encontró:** el **50,6 %** de las ventas lleva promoción, pero el ticket con promoción
(76 041) es **indistinguible** del ticket sin promoción (75 039). Ninguna de las tres mecánicas se
despega del grupo de control, y el `2x1` **tampoco mueve las unidades por venta**, que es
justamente lo que debería mover por diseño.

**Cómo se comprobó:** prueba t de Welch de cada promoción contra el grupo de control `sin_promo`.
Global **p = 0,65**; unidades del `2x1` **p = 0,55**. Ningún contraste se acerca a la significancia.

**Recomendación:** las promociones actuales **regalan margen a clientes que iban a comprar igual**.
A un descuento efectivo del 5 % sobre 1 518 ventas promocionadas, eso son **≈ 5,7 M anuales sin
retorno medible**. Hay que suspender la asignación indiscriminada y pasar a una **prueba A/B** con
grupo de control explícito, porque hoy las promos no se asignan bajo ningún criterio registrado y
por eso no se les puede atribuir ningún efecto.

### Reporte 4 — El medio de pago sí discrimina el ticket

In [ ]:
print(df.groupby('medio_pago')
        .agg(ventas=('id_venta','count'), ingresos=('total_venta','sum'),
             ticket=('total_venta','mean'))
        .sort_values('ticket', ascending=False).to_string())
print()
comparar('Transferencia vs efectivo',
         df.loc[df.medio_pago=='transferencia','total_venta'],
         df.loc[df.medio_pago=='efectivo','total_venta'], 'transferencia', 'efectivo')

sin = df.medio_pago=='no_informado'
print(f'\nVentas sin medio de pago registrado: {int(sin.sum())} '
      f'({df.loc[sin,"total_venta"].sum():,.0f} = {df.loc[sin,"total_venta"].sum()/df.total_venta.sum()*100:.1f}% de ingresos)')
comparar('Ticket: sin medio informado vs resto',
         df.loc[sin,'total_venta'], df.loc[~sin,'total_venta'], 'sin dato', 'con dato')

**Qué se encontró:** a diferencia de las promociones, el medio de pago **sí** se asocia al ticket:
la transferencia promedia **79 917** contra **71 835** del efectivo, **+11,3 %**. Aparte, **55 ventas
(4,4 M, el 2 % de los ingresos) quedaron sin medio de pago registrado**.

**Cómo se comprobó:** prueba t de Welch entre los extremos, **p = 0,019** (significativo). Sobre las
55 ventas sin dato: su ticket promedio es el más alto de la tabla, pero con tan pocos casos la
diferencia **no** es significativa (p = 0,53), así que **no** se puede afirmar que la falla de
captura se concentre en las ventas grandes.

**Recomendación:** (1) **piloto de incentivo por transferencia** en tickets altos — es el único
factor con asociación significativa de todo el laboratorio y además es el medio más barato para la
empresa, sin comisión de tarjeta, así que un 2-3 % de descuento se autofinancia. Ojo: asociación no
es causalidad, el cliente puede elegir transferencia *porque* la compra es grande, por eso es un
piloto medido y no un despliegue general. (2) **Hacer obligatorio el campo en el POS**: 4,4 M sin
medio de pago registrado impiden cuadrar caja contra los extractos, y sin ese dato el piloto
anterior no se puede medir.

---
## Conclusión

De los cuatro factores analizados como palancas de negocio, **tres resultaron no significativos**
(ticket por sucursal, promociones y afinidad entre productos) y el único con asociación real
—el medio de pago— viene con un hueco de captura que dificulta medirlo.

El hallazgo dominante del laboratorio **no es comercial sino de calidad de datos**: identificadores
de cliente sin valor de identidad, medio de pago ausente, errores de signo, marcadores `???` y ruido
de formato. Los últimos se pudieron corregir aquí; el del `id_cliente` no, porque el dato correcto
nunca se capturó. Antes de invertir en promociones o fidelización, la acción de mayor retorno es
**arreglar la captura de datos en el punto de venta**.